**MedRAG for Primary Care Medical QA**

Build MedRAG pipeline finetuned on MedMCQA comparing
1. QA without RAG
2. QA with RAG (StarPearls) - RAG on inference only
3. QA with RAG Augmented Finetuning - RAG on training *and* inference (RAFT [link text](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/raft-a-new-way-to-teach-llms-to-be-better-at-rag/4084674
))

_Steps_
  1. Load MedMCQA — identify primary care subjects, check dataset size, split into
  train/val/test
  2. Index StatPearls — FAISS + MedCPT embeddings
  3. Build retrieval helpers — wraps the index for use in later steps
  4. Zero-shot baseline — evaluate base PMC-LLaMA both without RAG and with RAG before any
  fine-tuning
  5. Fine-tune Model 1 — vanilla MedMCQA, no retrieved context
  6. Fine-tune Model 2 — RAG-augmented MedMCQA (RAFT-style)
  7. Evaluate all arms:
*   Model 1, no RAG
*   Model 1, with RAG (format mismatch condition)
*   Model 2, with RAG

  8. Failure analysis

Install Dependencies

In [1]:
!pip install peft bitsandbytes sentencepiece faiss-gpu-cu12 -q

Load and Split MedMCQA

In [2]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

dataset = load_dataset("openlifescienceai/medmcqa", split="train")

In [3]:
# Confirm the subject_names in the dataset

from collections import Counter
print(Counter(dataset['subject_name']).most_common(50))

[('Medicine', 17887), ('Surgery', 16862), ('Pathology', 14884), ('Anatomy', 14560), ('Pharmacology', 13758), ('Social & Preventive Medicine', 11882), ('Microbiology', 11314), ('Gynaecology & Obstetrics', 10013), ('Dental', 8938), ('Physiology', 8830), ('Biochemistry', 8282), ('Pediatrics', 8037), ('Ophthalmology', 6932), ('Forensic Medicine', 5900), ('ENT', 4919), ('Psychiatry', 4442), ('Radiology', 4395), ('Anaesthesia', 3172), ('Unknown', 3045), ('Orthopaedics', 2999), ('Skin', 1771)]


In [4]:
# Filter the medMCQ dataset with primary care subjects

PRIMARY_CARE_SUBJECTS = {
    "Medicine", "Pharmacology", "Microbiology",
    "Social & Preventive Medicine", "ENT", "Psychiatry",
    "Orthopaedics", "Skin"
}
# chose to exclude pediatrics, surgery, pathology, anatomy, gynaecology &
# obstetrics, detal, physiology, biochemistry, forensics, radiology & anasethesia

dataset = dataset.filter(lambda x: x["subject_name"] in PRIMARY_CARE_SUBJECTS)

# 80/20 split (use the official train split; val/test sets exist too)
split    = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]
test_ds  = split["test"]


Filter:   0%|          | 0/182822 [00:00<?, ? examples/s]

Build the StatPearls FAISS index with MedCPT

In [5]:
# Mount Google Drive to Store downloaded StatPearls data
# Warning ~2GB of space required

from google.colab import drive

drive.mount('/content/drive')
# This will prompt you to authenticate with your Google account.

# BASE_PATH = "/content/drive/MyDrive/BioNLP"  # Nick (comment out)
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks"  # Andrew - adjust to your path

Mounted at /content/drive


In [6]:
# Download NXML archive from NCBI FTP
# Ignore if chunking done
import subprocess
subprocess.run([
    "wget",
    "https://ftp.ncbi.nlm.nih.gov/pub/litarch/3d/12/statpearls_NBK430685.tar.gz",
    "-O", f"{BASE_PATH}/statpearls_NBK430685.tar.gz"
])

import tarfile
with tarfile.open(f"{BASE_PATH}/statpearls_NBK430685.tar.gz", "r:gz") as tar:
    tar.extractall(f"{BASE_PATH}/corpus/statpearls/")

/tmp/ipykernel_6339/2851299727.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(f"{BASE_PATH}/corpus/statpearls/")


In [ ]:
# Run MedRAG's chunking script. Clone the MedRAG repo and use their
# statpearls.py to convert the NXML files into the chunk.jsonl format your original
# code expects:
# Ingore if chunking done
import os
os.chdir(BASE_PATH)
!python "/content/drive/MyDrive/Spring 2026/CBB 5740 HW3/MedRAG-main/src/data/statpearls.py"

 98% 9445/9638 [06:21<00:04, 41.36it/s]

In [ ]:
# Merge chunked .jsonl files in chunk/ folder into a single statpearls.jsonl file
# Ignore if chunking done
import glob

chunk_files = sorted(glob.glob(f"{BASE_PATH}/corpus/statpearls/chunk/*.jsonl"))
with open(f"{BASE_PATH}/statpearls.jsonl", "w") as out:
    for f in chunk_files:
        with open(f) as inp:
            content = inp.read()
            out.write(content)
            if not content.endswith('\n'):
                out.write('\n')

print("Done. Total chunks:", sum(1 for _ in open(f"{BASE_PATH}/statpearls.jsonl")))

Done. Total chunks: 365170


In [ ]:
import json, torch, faiss
import numpy as np
from transformers import AutoTokenizer, AutoModel
from google.colab import userdata

# --- Load MedCPT article encoder ---
ENC_MODEL = "ncbi/MedCPT-Article-Encoder"

hf_token = userdata.get('HF_TOKEN')
tok = AutoTokenizer.from_pretrained(ENC_MODEL, token=hf_token)
enc = AutoModel.from_pretrained(ENC_MODEL, token=hf_token).cuda().eval()

# --- Load StatPearls corpus from Drive ---
with open(f"{BASE_PATH}/statpearls.jsonl") as f:
    corpus = [json.loads(l) for l in f]

print("Corpus size:", len(corpus))

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Corpus size: 365170


In [ ]:
# Clean up statpearls: delete unnecessary files

import subprocess
result = subprocess.run(
    ["rm", "-rf", f"{BASE_PATH}/corpus/statpearls/statpearls_NBK430685"],
    capture_output=True, text=True
)
print("Done.", result.returncode)
#Also delete the tar.gz at the same time:
'''
import os
os.remove(f"{BASE_PATH}/statpearls_NBK430685.tar.gz")
print("Deleted tar.gz")'''

KeyboardInterrupt: 

In [ ]:
# --- check to see if statpearls exists

index = faiss.read_index(f"{BASE_PATH}/statpearls.faiss")
print("Index loaded:", index.ntotal, "chunks")

Index loaded: 365170 chunks


In [ ]:
# --- Encode corpus chunks ---
# Ignore if above cell confirms chunks loaded

BATCH = 64

def encode_texts(batch_texts):
    inputs = tok(batch_texts, return_tensors="pt",
                  padding=True, truncation=True,
                  max_length=512).to("cuda")
    with torch.no_grad():
        out = enc(**inputs)
    return out.last_hidden_state[:, 0, :].cpu().float().numpy()

all_embs = []
for i in range(0, len(corpus), BATCH):
    batch = [c["content"] for c in corpus[i:i+BATCH]]
    all_embs.append(encode_texts(batch))
    if i % 1000 == 0:
        print(f"Encoded {i}/{len(corpus)}")

embeddings = np.vstack(all_embs)

# --- Build FAISS index ---
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
faiss.normalize_L2(embeddings)
index.add(embeddings)
faiss.write_index(index, f"{BASE_PATH}/statpearls.faiss")
print("Index built:", index.ntotal, "chunks")

Encoded 0/365170
Encoded 8000/365170
Encoded 16000/365170
Encoded 24000/365170
Encoded 32000/365170
Encoded 40000/365170
Encoded 48000/365170
Encoded 56000/365170
Encoded 64000/365170
Encoded 72000/365170
Encoded 80000/365170
Encoded 88000/365170
Encoded 96000/365170
Encoded 104000/365170
Encoded 112000/365170
Encoded 120000/365170
Encoded 128000/365170
Encoded 136000/365170
Encoded 144000/365170
Encoded 152000/365170
Encoded 160000/365170
Encoded 168000/365170
Encoded 176000/365170
Encoded 184000/365170
Encoded 192000/365170
Encoded 200000/365170
Encoded 208000/365170
Encoded 216000/365170
Encoded 224000/365170
Encoded 232000/365170
Encoded 240000/365170
Encoded 248000/365170
Encoded 256000/365170
Encoded 264000/365170
Encoded 272000/365170
Encoded 280000/365170
Encoded 288000/365170
Encoded 296000/365170
Encoded 304000/365170
Encoded 312000/365170
Encoded 320000/365170
Encoded 328000/365170
Encoded 336000/365170
Encoded 344000/365170
Encoded 352000/365170
Encoded 360000/365170
Index 

Retrieval Helper

In [ ]:
QUERY_ENC = "ncbi/MedCPT-Query-Encoder"
q_tok = AutoTokenizer.from_pretrained(QUERY_ENC, token=hf_token)
q_enc = AutoModel.from_pretrained(QUERY_ENC, token=hf_token).cuda().eval()

def retrieve(question, k=5):
    inputs = q_tok(question, return_tensors="pt",
                    padding=True, truncation=True,
                    max_length=512).to("cuda")
    with torch.no_grad():
        q_emb = q_enc(**inputs).last_hidden_state[:, 0, :].cpu().float().numpy()
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    return [corpus[i]["content"] for i in idxs[0]]

def build_prompt_rag(example, context_chunks):
    ctx = "\n\n".join(context_chunks)
    opts = (f"A. {example['opa']}\nB. {example['opb']}\n"
            f"C. {example['opc']}\nD. {example['opd']}")
    return (
        f"Context:\n{ctx}\n\n"
        f"Question: {example['question']}\n{opts}\n\n"
        f"Answer with A, B, C, or D and a brief explanation.\nAnswer:"
    )

def build_prompt_norag(example):
    opts = (f"A. {example['opa']}\nB. {example['opb']}\n"
            f"C. {example['opc']}\nD. {example['opd']}")
    return (
        f"Question: {example['question']}\n{opts}\n\n"
        f"Answer with A, B, C, or D and a brief explanation.\nAnswer:"
    )

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Load PMC-LLaMA (4-bit quantized for Colab)

In [ ]:
# Clear cache before rerunning

import torch, gc
gc.collect()
torch.cuda.empty_cache()

Changed the below to BioMistral from PMC LLAMA because it kept on running out of recursion depth.

In [ ]:
# Load base PMC-LLaMA without any adapters.
from transformers import LlamaTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

PMC_MODEL = "BioMistral/BioMistral-7B"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

llm_tok = AutoTokenizer.from_pretrained(PMC_MODEL,
token=hf_token)
llm_tok.pad_token = llm_tok.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    PMC_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
    token=hf_token,
    use_safetensors=False,
)

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The repo does no

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Run zero-shot baseline evaluation

In [ ]:
import re
from tqdm import tqdm

# MedMCQA cop field is 1-indexed: 1=A, 2=B, 3=C, 4=D
COP_TO_LETTER = {1: "A", 2: "B", 3: "C", 4: "D"}

def generate_answer(prompt, max_new_tokens=100):
    inputs = llm_tok(prompt, return_tensors="pt",
                      truncation=True, max_length=1024).to(llm.device)
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tok.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return llm_tok.decode(new_tokens, skip_special_tokens=True)

def parse_answer(text):
    match = re.search(r'\b([ABCD])\b', text)
    return match.group(1) if match else None

In [ ]:
COP_TO_LETTER = {0: "A", 1: "B", 2: "C", 3: "D"}
eval_subset = test_ds.select(range(500))

# --- Zero-shot, no RAG ---
norag_preds, norag_labels = [], []
for ex in tqdm(eval_subset, desc="Zero-shot no RAG"):
    prompt = build_prompt_norag(ex)
    pred = parse_answer(generate_answer(prompt))
    norag_preds.append(pred)
    norag_labels.append(COP_TO_LETTER[ex["cop"]])

norag_acc = (sum(p == l for p, l in zip(norag_preds, norag_labels) if p is not None) /
              len(norag_labels))
print(f"Zero-shot no RAG accuracy: {norag_acc:.3f}")

# --- Zero-shot, with RAG ---
rag_preds, rag_labels = [], []
for ex in tqdm(eval_subset, desc="Zero-shot with RAG"):
    chunks = retrieve(ex["question"])
    prompt = build_prompt_rag(ex, chunks)
    pred = parse_answer(generate_answer(prompt))
    rag_preds.append(pred)
    rag_labels.append(COP_TO_LETTER[ex["cop"]])

rag_acc = (sum(p == l for p, l in zip(rag_preds, rag_labels) if p is not None) /
            len(rag_labels))
print(f"Zero-shot with RAG accuracy: {rag_acc:.3f}")

NameError: name 'test_ds' is not defined

This is where I got to - Above this lines has been tested and works

In [ ]:
# --- Save results to Drive ---
results = {
    "zero_shot_norag_accuracy": norag_acc,
    "zero_shot_rag_accuracy": rag_acc,
    "norag_predictions": norag_preds,
    "rag_predictions": rag_preds,
    "labels": norag_labels,
}
with open(f"{BASE_PATH}/baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved:", f"{BASE_PATH}/baseline_results.json")

Fine tuning

Train model 1: Vanilla MedMCQA

In [ ]:
from torch.utils.data import Dataset

class VanillaDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=1024):
        self.data      = hf_dataset
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.label_map = {0: "A", 1: "B", 2: "C", 3: "D"}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex      = self.data[idx]
        prompt  = build_prompt_norag(ex)
        answer  = self.label_map[ex["cop"]]
        full    = prompt + " " + answer + llm_tok.eos_token

        enc = self.tokenizer(
            full, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        labels = enc["input_ids"].clone()
        prompt_len = len(self.tokenizer(prompt)["input_ids"])
        labels[0, :prompt_len] = -100

        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         labels.squeeze(),
        }

vanilla_train_dataset = VanillaDataset(train_ds, llm_tok)

Vanilla model fine tuning

In [ ]:
from transformers import TrainingArguments, Trainer,
DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType
import gc, torch

def make_lora_cfg():
    return LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

def make_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,                          # match
bnb_4bit_compute_dtype=bfloat16
        logging_steps=50,
        save_strategy="epoch",
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        report_to="none",
    )

In [ ]:
# --- Model 1: Vanilla (no RAG) ---
llm = get_peft_model(llm, make_lora_cfg())
llm.print_trainable_parameters()

Trainer(
    model=llm,
    args=make_args(f"{BASE_PATH}/model1-vanilla"),
    train_dataset=vanilla_train_dataset,
    data_collator=DataCollatorForSeq2Seq(llm_tok,
model=llm, padding=True),
).train()

llm.save_pretrained(f"{BASE_PATH}/model1-vanilla-final
")
print("Model 1 saved")

Train model 2: RAFT

Build a RAG-augmented dataset for fine-tuning

In [ ]:
from torch.utils.data import Dataset

class MedRAGDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, k=5, max_len=2048):
        self.data      = hf_dataset
        self.tokenizer = tokenizer
        self.k         = k
        self.max_len   = max_len
        self.label_map = {0: "A", 1: "B", 2: "C", 3: "D"} # check this

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex      = self.data[idx]
        chunks  = retrieve(ex["question"], k=self.k)
        prompt  = build_prompt_rag(ex, chunks)
        answer  = self.label_map[ex["cop"]]
        full    = prompt + " " + answer + llm_tok.eos_token

        enc = self.tokenizer(
            full, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        labels = enc["input_ids"].clone()
        # Mask the prompt tokens — only train on the answer
        prompt_len = len(self.tokenizer(prompt)["input_ids"])
        labels[0, :prompt_len] = -100

        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         labels.squeeze(),
        }

train_dataset = MedRAGDataset(train_ds, llm_tok)

Fine-tune with Trainer

In [ ]:
 # --- Reset for Model 2 ---
  del llm
  gc.collect()
  torch.cuda.empty_cache()

  llm = AutoModelForCausalLM.from_pretrained(
      PMC_MODEL,
      quantization_config=bnb_cfg,
      device_map="auto",
      token=hf_token,
  )

In [ ]:
  # --- Model 2: RAG-augmented (RAFT) ---
  llm = get_peft_model(llm, make_lora_cfg())
  llm.print_trainable_parameters()

  Trainer(
      model=llm,
      args=make_args(f"{BASE_PATH}/model2-raft"),
      train_dataset=train_dataset,            #
  MedRAGDataset with retrieved context
      data_collator=DataCollatorForSeq2Seq(llm_tok,
  model=llm, padding=True),
  ).train()

  llm.save_pretrained(f"{BASE_PATH}/model2-raft-final")
  print("Model 2 saved")

Evaluation

In [ ]:
from peft import PeftModel
import json

# Load baseline results saved earlier
with open(f"{BASE_PATH}/baseline_results.json") as f:
    baseline = json.load(f)

def eval_model(dataset, use_rag=False, desc=""):
    preds, labels = [], []
    for ex in tqdm(dataset, desc=desc):
        if use_rag:
            chunks = retrieve(ex["question"])
            prompt = build_prompt_rag(ex, chunks)
        else:
            prompt = build_prompt_norag(ex)
        pred = parse_answer(generate_answer(prompt))
        preds.append(pred)
        labels.append(COP_TO_LETTER[ex["cop"]])
    acc = sum(p == l for p, l in zip(preds, labels) if
  p is not None) / len(labels)
    return acc, preds

# --- Load and evaluate Model 1 (vanilla, no RAG training) ---
del llm
gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    PMC_MODEL, quantization_config=bnb_cfg,
device_map="auto", token=hf_token
)
llm = PeftModel.from_pretrained(base, f"{BASE_PATH}/model1-vanilla-final").eval()

m1_norag_acc, _ = eval_model(test_ds, use_rag=False, desc="Model 1 no-RAG")
print(f"Model 1 no-RAG: {m1_norag_acc:.3f}")

m1_rag_acc, _   = eval_model(test_ds, use_rag=True, desc="Model 1 with RAG (format mismatch)")
print(f"Model 1 RAG (format mismatch):
{m1_rag_acc:.3f}")

# --- Load and evaluate Model 2 (RAFT) ---
del llm, base
gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    PMC_MODEL, quantization_config=bnb_cfg,
device_map="auto", token=hf_token
)
llm = PeftModel.from_pretrained(base,
f"{BASE_PATH}/model2-raft-final").eval()

m2_rag_acc, _   = eval_model(test_ds, use_rag=True,
desc="Model 2 RAG (RAFT)")
print(f"Model 2 RAG (RAFT): {m2_rag_acc:.3f}")

# --- Results table ---
results = {
    "Model 0 no-RAG (zero-shot)":
baseline["zero_shot_norag_accuracy"],
    "Model 0 RAG (zero-shot)":
baseline["zero_shot_rag_accuracy"],
    "Model 1 no-RAG (vanilla FT)":     m1_norag_acc,
    "Model 1 RAG (format mismatch)":   m1_rag_acc,
    "Model 2 RAG (RAFT)":              m2_rag_acc,
}

print("\n{:<35} {:>8}".format("Condition",
"Accuracy"))
print("-" * 45)
for name, acc in results.items():
    print("{:<35} {:>8.3f}".format(name, acc))

# Save to Drive
with open(f"{BASE_PATH}/final_results.json", "w") as
f:
    json.dump(results, f, indent=2)
print("\nSaved to Drive")

SyntaxError: unterminated f-string literal (detected at line 38) (1105947426.py, line 38)

References:

In [ ]:
@article{jin2023medcpt,
  title={MedCPT: Contrastive Pre-trained Transformers with large-scale PubMed search logs for zero-shot biomedical information retrieval},
  author={Jin, Qiao and Kim, Won and Chen, Qingyu and Comeau, Donald C and Yeganova, Lana and Wilbur, W John and Lu, Zhiyong},
  journal={Bioinformatics},
  volume={39},
  number={11},
  pages={btad651},
  year={2023},
  publisher={Oxford University Press}
}